# Transformer-XL: Mathematical Aspects of Key Improvements

Transformer-XL introduces several significant improvements over the original Transformer model, making it more efficient and capable of handling longer context dependencies in sequential data. Here are the key new features and enhancements in Transformer-XL:

## 1. Segment-Level Recurrence

**Original Transformer**:
In the original Transformer, the sequence is divided into fixed-length segments, and the model processes each segment independently. This limits the model's ability to capture dependencies beyond the segment length.

**Transformer-XL**:
Transformer-XL introduces a segment-level recurrence mechanism, where hidden states from the previous segment are carried over to the current segment. Mathematically, this is expressed as:

$\mathbf{H}_t = \text{Transformer}(\mathbf{X}_t, \mathbf{M}_{t-1})$

where:
- $\mathbf{H}_t$ are the hidden states for the current segment $t$.
- $\mathbf{X}_t$ is the input for the current segment $t$.
- $\mathbf{M}_{t-1}$ are the hidden states from the previous segment $t-1$, which act as memory.

This recurrence allows the model to retain information across segment boundaries, effectively extending the context length.

## 2. Relative Positional Encodings

**Absolute Positional Encodings**:
In the original Transformer, positional encodings are added to the input embeddings to provide information about the position of tokens. For a sequence position $i$, the positional encoding is given by:

$\text{PE}(i, 2k) = \sin\left(\frac{i}{10000^{2k/d}}\right)$
$\text{PE}(i, 2k+1) = \cos\left(\frac{i}{10000^{2k/d}}\right)$

where $d$ is the dimensionality of the embeddings.

**Relative Positional Encodings**:
Transformer-XL uses relative positional encodings, which consider the relative distance between tokens rather than their absolute positions. This allows the model to generalize better across different lengths. For a pair of positions $i$ and $j$ in the sequence, the relative positional encoding $\mathbf{r}_{i-j}$ is used, modifying the self-attention mechanism:

$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T + Q\mathbf{R}^T_{i-j}}{\sqrt{d}}\right)V$

where $Q$ (queries), $K$ (keys), and $V$ (values) are the usual components in self-attention, and $\mathbf{R}_{i-j}$ represents the relative positional encoding for the distance $i-j$.

## 3. Memory Mechanism

In Transformer-XL, the memory mechanism is implemented by maintaining a memory of the hidden states from previous segments. This memory is updated at each segment:

$\mathbf{M}_t = \text{SG}(\text{concat}(\mathbf{M}_{t-1}, \mathbf{H}_t))$

where:
- $\mathbf{M}_t$ is the memory after processing segment $t$.
- $\text{SG}$ denotes stop-gradient, meaning the gradients are not backpropagated through the memory from previous segments.

The concatenation of $\mathbf{M}_{t-1}$ and $\mathbf{H}_t$ allows the model to use a longer context for the next segment, significantly extending the effective context length.

## 4. Reduced Training Complexity

By reusing hidden states and employing the memory mechanism, Transformer-XL reduces the training complexity. In a standard Transformer, the computational complexity for a sequence of length $n$ is $O(n^2)$ due to the self-attention mechanism. In Transformer-XL, the segment-level recurrence and memory mechanism reduce the effective sequence length processed at each step, leading to more efficient computation.

## 5. State-of-the-Art Performance

The improvements in handling long-range dependencies, efficient memory usage, and better generalization capabilities have led Transformer-XL to achieve state-of-the-art performance on various benchmarks. For example, in language modeling tasks, the model achieves lower perplexity scores, indicating better predictive performance.

## 6. Improved Generalization

The use of relative positional encodings and segment-level recurrence allows Transformer-XL to generalize better to longer sequences. The model is not tied to fixed segment lengths and can adapt to varying sequence lengths during inference, providing more robust and versatile performance in practical applications.

By incorporating these mathematical innovations, Transformer-XL addresses the limitations of the original Transformer model and significantly enhances its ability to handle long-term dependencies and large-scale sequential data.

## References

1. Dai, Z., Yang, Z., Yang, Y., Carbonell, J., Le, Q. V., & Salakhutdinov, R. (2019). Transformer-XL: Attentive Language Models Beyond a Fixed-Length Context. Retrieved from [https://arxiv.org/abs/1901.02860](https://arxiv.org/abs/1901.02860)
2. Vaswani, A., Shazeer, N., Parmar, N., Uszkoreit, J., Jones, L., Gomez, A. N., ... & Polosukhin, I. (2017). Attention is all you need. Retrieved from [https://arxiv.org/abs/1706.03762](https://arxiv.org/abs/1706.03762)

In [6]:
!pip install torch==2.0.1+cu117 torchvision==0.15.2+cu117 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu117

Looking in indexes: https://download.pytorch.org/whl/cu117
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 GB 846.5 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 82.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 27.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.3/63.3 MB 10.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.3/132.3 kB 10.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for lit: filename=lit-15.0.7-py3-none-any.whl size=89990 sha256=48c54b51f40123e900badbab96a1d1d22e7e1ceb41a4b5e66f76da2abba72618
  Stored in directory: /root/.cache/pip/wheels/27/2c/b6/3ed2983b1b44fe0dea1bb35234b09f2c22fb8ebb308679c922
Successfully built lit
  Attempting uninstall: triton
    Found existing installation: triton 2.3.1
    Uninstalling triton-2.3.1:
      Successfully uninstalled triton-2.3.1
  Attempting uninstall: torch
    Found existing installation: torch 2.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class PositionalEmbedding(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model
        inv_freq = 1 / (10000 ** (torch.arange(0.0, d_model, 2.0) / d_model))
        self.register_buffer('inv_freq', inv_freq)

    def forward(self, pos_seq):
        sinusoid_inp = torch.outer(pos_seq, self.inv_freq)
        pos_emb = torch.cat([sinusoid_inp.sin(), sinusoid_inp.cos()], dim=-1)
        return pos_emb

class RelativeMultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_head, d_head, dropout):
        super().__init__()
        self.d_model = d_model
        self.n_head = n_head
        self.d_head = d_head
        self.dropout = dropout

        self.q_net = nn.Linear(d_model, n_head * d_head, bias=False)
        self.k_net = nn.Linear(d_model, n_head * d_head, bias=False)
        self.v_net = nn.Linear(d_model, n_head * d_head, bias=False)

        self.r_net = nn.Linear(d_model, n_head * d_head, bias=False)
        self.o_net = nn.Linear(n_head * d_head, d_model, bias=False)

        self.drop = nn.Dropout(dropout)
        self.scale = 1 / (d_head ** 0.5)

    def _rel_shift(self, x):
        zero_pad = torch.zeros((x.size(0), 1, *x.size()[2:]), device=x.device, dtype=x.dtype)
        x_padded = torch.cat([zero_pad, x], dim=1)
        x_padded = x_padded.view(x.size(1) + 1, x.size(0), *x.size()[2:])
        x = x_padded[1:].view_as(x)
        return x

    def forward(self, w, r, attn_mask=None, mems=None):
        qlen, bsz = w.size(0), w.size(1)
        rlen = r.size(0)

        if mems is not None:
            cat = torch.cat([mems, w], 0)
            w_heads = self.k_net(cat)
            w_head_v = self.v_net(cat)
            mlen = mems.size(0)
        else:
            mlen = 0
            w_heads = self.k_net(w)
            w_head_v = self.v_net(w)

        klen = w_heads.size(0)

        w_head_q = self.q_net(w)
        r_head_k = self.r_net(r)

        w_head_q = w_head_q.view(qlen, bsz, self.n_head, self.d_head)
        w_heads = w_heads.view(klen, bsz, self.n_head, self.d_head)
        w_head_v = w_head_v.view(klen, bsz, self.n_head, self.d_head)
        r_head_k = r_head_k.view(rlen, self.n_head, self.d_head)

        # Ensure r_head_k has the correct shape
        r_head_k = r_head_k.unsqueeze(1).expand(-1, bsz, -1, -1)

        # Adjust rw_head_q calculation
        rw_head_q = w_head_q + r_head_k[-qlen:]

        # Adjust AC and BD calculations
        AC = torch.einsum('ibnd,jbnd->ijbn', (rw_head_q, w_heads))
        BD = torch.einsum('ibnd,jbnd->ijbn', (w_head_q, r_head_k))
        BD = self._rel_shift(BD)

        attn_score = AC + BD
        attn_score.mul_(self.scale)

        if attn_mask is not None:
            attn_score = attn_score.float().masked_fill(
                attn_mask[None, :, :, None], float('-inf')).type_as(attn_score)

        attn_prob = F.softmax(attn_score, dim=1)
        attn_prob = self.drop(attn_prob)

        attn_vec = torch.einsum('ijbn,jbnd->ibnd', (attn_prob, w_head_v))
        attn_vec = attn_vec.contiguous().view(qlen, bsz, self.n_head * self.d_head)

        attn_out = self.o_net(attn_vec)
        attn_out = self.drop(attn_out)

        return attn_out

class TransformerXLLayer(nn.Module):
    def __init__(self, d_model, n_head, d_head, d_inner, dropout):
        super().__init__()
        self.dec_attn = RelativeMultiHeadAttention(d_model, n_head, d_head, dropout)
        self.pos_ff = nn.Sequential(
            nn.Linear(d_model, d_inner),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(d_inner, d_model),
            nn.Dropout(dropout),
        )
        self.layer_norm1 = nn.LayerNorm(d_model)
        self.layer_norm2 = nn.LayerNorm(d_model)

    def forward(self, dec_inp, r, dec_attn_mask=None, mems=None):
        attn_out = self.dec_attn(dec_inp, r, attn_mask=dec_attn_mask, mems=mems)
        attn_out = self.layer_norm1(dec_inp + attn_out)
        ff_out = self.pos_ff(attn_out)
        ff_out = self.layer_norm2(attn_out + ff_out)
        return ff_out

class TransformerXL(nn.Module):
    def __init__(self, n_token, n_layer, n_head, d_model, d_head, d_inner, dropout, mem_len):
        super().__init__()
        self.d_model = d_model
        self.n_head = n_head
        self.d_head = d_head
        self.mem_len = mem_len
        self.n_layer = n_layer

        self.word_emb = nn.Embedding(n_token, d_model)
        self.pos_emb = PositionalEmbedding(d_model)
        self.drop = nn.Dropout(dropout)

        self.layers = nn.ModuleList()
        for _ in range(n_layer):
            self.layers.append(
                TransformerXLLayer(d_model, n_head, d_head, d_inner, dropout)
            )

    def _update_mems(self, hids, mems, qlen, mlen):
        if mems is None:
            mems = [None] * self.n_layer
        new_mems = []
        for i in range(self.n_layer):
            cat = torch.cat([mems[i] if mems[i] is not None else torch.empty(0, device=hids[i].device), hids[i]], dim=0)
            new_mems.append(cat[-mlen:].detach())
        return new_mems

    def forward(self, inp, mems=None):
        qlen, bsz = inp.size()

        if mems is None:
            mems = [None] * self.n_layer

        word_emb = self.word_emb(inp)

        mlen = mems[0].size(0) if mems[0] is not None else 0
        klen = mlen + qlen

        pos_seq = torch.arange(klen - 1, -1, -1.0, device=word_emb.device, dtype=word_emb.dtype)
        pos_emb = self.pos_emb(pos_seq)

        core_out = self.drop(word_emb)
        pos_emb = self.drop(pos_emb)

        hids = []
        for i, layer in enumerate(self.layers):
            hids.append(core_out)
            mems_i = mems[i] if mems[i] is not None else None
            core_out = layer(core_out, pos_emb, mems=mems_i)

        core_out = self.drop(core_out)

        new_mems = self._update_mems(hids, mems, qlen, self.mem_len)
        return core_out, new_mems

# Example usage
vocab_size = 10000
n_layer = 4
n_head = 8
d_model = 512
d_head = 64
d_inner = 2048
dropout = 0.1
mem_len = 512

model = TransformerXL(vocab_size, n_layer, n_head, d_model, d_head, d_inner, dropout, mem_len)

# Use torch.compile() for potential performance improvements
model = torch.compile(model)

# Generate some dummy data
seq_len = 64
batch_size = 32
input_seq = torch.randint(0, vocab_size, (seq_len, batch_size))

# Initialize memory
mems = [torch.zeros(mem_len, batch_size, d_model) for _ in range(n_layer)]

# Forward pass
output, new_mems = model(input_seq, mems)

print(f"Output shape: {output.shape}")
print(f"Number of memory tensors: {len(new_mems)}")
print(f"Memory tensor shape: {new_mems[0].shape}")

# Optional: run a second time to see compiled model performance
output, new_mems = model(input_seq, new_mems)

No CUDA runtime is found, using CUDA_HOME='/usr/local/cuda'


Output shape: torch.Size([64, 32, 512])
Number of memory tensors: 4
Memory tensor shape: torch.Size([512, 32, 512])
